# Movie Recommendation System Using Collaborative Filtering

**Course**: Data Analysis & Management  
**Dataset**: MovieLens (Kaggle - The Movies Dataset)  
**Objective**: Implement and evaluate collaborative filtering algorithms for movie recommendations

---

## Project Overview

This notebook implements a comprehensive movie recommendation system using the MovieLens dataset. The project follows these major phases:

1. **Data Setup** - Install dependencies and prepare environment
2. **Dataset Acquisition** - Load and understand the MovieLens dataset structure
3. **Data Preprocessing** - Clean and prepare all datasets (ratings, metadata, credits, keywords, links)
4. **Train/Test Split** - Create training and test sets using random and temporal split methods
5. **Model Implementation** - Build collaborative filtering algorithms (covered in separate notebooks)
6. **Evaluation** - Assess model performance using RMSE, MAE, Precision@K metrics

---

# Data Setup

Install required libraries for data processing and collaborative filtering implementation.

**Note for macOS users**: If you see SSL/OpenSSL warnings, you can safely ignore them or suppress with:
```python
import warnings
warnings.filterwarnings('ignore', category=Warning)
```

In [41]:
# Install required libraries
!pip install pandas numpy scikit-learn scipy matplotlib seaborn
!pip install scikit-surprise  # For collaborative filtering algorithms
!pip install kagglehub  # For downloading datasets from Kaggle

zsh:1: command not found: pip
zsh:1: command not found: pip
zsh:1: command not found: pip


## Notebook Setup

Import necessary libraries for data manipulation, analysis, and visualization.

In [3]:
# Import core libraries
import pandas as pd
import numpy as np
import ast
import os
import glob
import shutil
import warnings
from datetime import datetime

# Suppress SSL and other warnings (optional - for cleaner output)
warnings.filterwarnings('ignore', category=Warning)

# Data splitting
from sklearn.model_selection import train_test_split

# Kaggle dataset downloader
import kagglehub

# Visualization (optional)
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

print(" All libraries imported successfully")

 All libraries imported successfully


# Dataset

## Dataset Overview

We use the **MovieLens dataset** from Kaggle ([The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset)).

### Dataset Structure

| File | Size | Rows | Description |
|------|------|------|-------------|
| `ratings.csv` | 710MB | ~26M | User-movie ratings (userId, movieId, rating, timestamp) |
| `movies_metadata.csv` | 34MB | ~45K | Movie details with TMDB data |
| `credits.csv` | 190MB | ~45K | Cast and crew information (JSON) |
| `keywords.csv` | 6MB | ~46K | Movie keywords/tags (JSON) |
| `links.csv` | 989KB | ~45K | ID mappings (movieId ↔ imdbId ↔ tmdbId) |

### Fetch Dataset from Kaggle

This notebook automatically downloads the dataset using `kagglehub` and stores it in `data/raw/`.

```

In [4]:
# Download dataset from Kaggle to local data folder
print("Downloading MovieLens dataset from Kaggle...")

# Define local data directories (relative to notebook location)
RAW_DATA_DIR = './data/raw/'
PROCESSED_DATA_DIR = './data/processed/'

# Create directories if they don't exist
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
print(f"Data directories verified:")
print(f"  - Raw data: {RAW_DATA_DIR}")
print(f"  - Processed data: {PROCESSED_DATA_DIR}\n")

# Check if dataset already exists locally
required_files = ['ratings.csv', 'movies_metadata.csv', 'credits.csv', 'keywords.csv', 'links.csv']
all_files_exist = all(os.path.exists(os.path.join(RAW_DATA_DIR, f)) for f in required_files)

if all_files_exist:
    print("Dataset already exists locally. Skipping download.\n")
    DATASET_PATH = RAW_DATA_DIR
else:
    print("Downloading from Kaggle...")
    
    # Download latest version to cache first
    cache_path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")
    
    # Copy files from cache to local raw data directory
    cache_files = glob.glob(os.path.join(cache_path, "*.csv"))
    
    print(f"Copying files to {RAW_DATA_DIR}...\n")
    for cache_file in cache_files:
        file_name = os.path.basename(cache_file)
        dest_path = os.path.join(RAW_DATA_DIR, file_name)
        shutil.copy2(cache_file, dest_path)
        print(f"  Copied {file_name}")
    
    DATASET_PATH = RAW_DATA_DIR
    print(f"\nDataset downloaded and saved to: {DATASET_PATH}")

# List files in raw data directory
print("\nDataset files in raw directory:")
for file in sorted(required_files):
    filepath = os.path.join(DATASET_PATH, file)
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"  {file:30} ({size_mb:.2f} MB)")

# Store the path for later use
print(f"\nUsing dataset path: {DATASET_PATH}")

Data directories verified:
  - Raw data: ./data/raw/
  - Processed data: ./data/processed/

Dataset already exists locally. Skipping download.


Dataset files in raw directory:
  credits.csv                    (181.12 MB)
  keywords.csv                   (5.94 MB)
  links.csv                      (0.94 MB)
  movies_metadata.csv            (32.85 MB)
  ratings.csv                    (676.68 MB)

Using dataset path: ./data/raw/


# Data Preprocessing

This section consolidates all data cleaning and preparation steps from the individual notebooks in `sources/1_data_preparation/`.

**Processing Order**:
1. Ratings (core dataset)
2. Movies Metadata (enrichment)
3. Links (ID mapping)
4. Credits (cast/crew - optional)
5. Keywords (tags - optional)

---

## 1. Clean Ratings Dataset

The ratings dataset is the **core dataset** for collaborative filtering. It contains:
- `userId`: User identifier
- `movieId`: Movie identifier (MovieLens internal ID)
- `rating`: Rating value (0.5 to 5.0, in 0.5 increments)
- `timestamp`: Unix timestamp when rating was made

### Data Quality Checks
- Check for missing values
- Check for duplicate user-movie pairs
- Verify rating range (0.5 - 5.0)

In [5]:
# Load ratings dataset
print("Loading ratings dataset...")
ratings_df = pd.read_csv(os.path.join(DATASET_PATH, 'ratings.csv'), low_memory=False)
ratings = ratings_df.copy()

print(f"Dataset loaded successfully")
print(f"\nShape: {ratings.shape}")
print(f"Columns: {list(ratings.columns)}")
print(f"\nSample data:")
print(ratings.head())

Loading ratings dataset...
Dataset loaded successfully

Shape: (26024289, 4)
Columns: ['userId', 'movieId', 'rating', 'timestamp']

Sample data:
   userId  movieId  rating   timestamp
0       1      110     1.0  1425941529
1       1      147     4.5  1425942435
2       1      858     5.0  1425941523
3       1     1221     5.0  1425941546
4       1     1246     5.0  1425941556


In [45]:
# Check for missing data
print("Columns with missing data:\n")
missing_data = []

for column in ratings.columns:
    missing_count = ratings[column].isnull().sum()
    if missing_count > 0:
        missing_pct = (missing_count / len(ratings)) * 100
        missing_data.append({
            'Column': column,
            'Missing count': missing_count,
            'Missing percentage': f"{missing_pct:.2f}%",
        })
        print(f"{column:30} | {missing_count} missing {missing_pct:5.2f}%")

if len(missing_data) == 0:
    print("No missing data found")
    
print(f"\nTotal columns: {len(ratings.columns)}")
print(f"Columns with missing data: {len(missing_data)}")
print(f"Columns without missing data: {len(ratings.columns) - len(missing_data)}")

Columns with missing data:

No missing data found

Total columns: 4
Columns with missing data: 0
Columns without missing data: 4


In [46]:
# Check for duplicate user-movie pairs
duplicates = ratings.groupby(['userId', 'movieId']).size()
duplicate_pairs = duplicates[duplicates > 1]

print(f"Total ratings: {len(ratings):,}")
print(f"Unique user-movie pairs: {len(duplicates):,}")
print(f"Duplicate user-movie pairs: {len(duplicate_pairs):,}")

if len(duplicate_pairs) > 0:
    duplicate_percentage = (len(duplicate_pairs) / len(duplicates)) * 100
    print(f"Percentage of duplicates: {duplicate_percentage:.2f}%")
    
    # Keep most recent rating if duplicates exist
    print("\nRemoving duplicates (keeping most recent rating)...")
    ratings = ratings.sort_values('timestamp').drop_duplicates(
        subset=['userId', 'movieId'],
        keep='last'
    )
    print(f"Removed {len(ratings_df) - len(ratings):,} duplicate ratings")
else:
    print("\n No duplicate ratings found")

Total ratings: 26,024,289
Unique user-movie pairs: 26,024,289
Duplicate user-movie pairs: 0

 No duplicate ratings found


In [47]:
# Save cleaned ratings
output_path = os.path.join(PROCESSED_DATA_DIR, 'cleaned_ratings.csv')
ratings.to_csv(output_path, index=False)

print(f" Cleaned ratings saved to: {output_path}")
print(f"Final dataset: {len(ratings):,} rows, {len(ratings.columns)} columns")

 Cleaned ratings saved to: ./data/processed/cleaned_ratings.csv
Final dataset: 26,024,289 rows, 4 columns


## 2. Clean Movies Metadata

The movies metadata provides rich information about each movie:
- Movie titles, release dates, runtime
- Genres, keywords (stored as JSON)
- Budget, revenue, popularity
- Vote average and count

### Cleaning Steps
1. Fix malformed `id` field (some rows have mixed types)
2. Drop rows with missing critical fields (id, title)
3. Parse JSON columns (genres, production_companies, etc.)
4. Keep only relevant columns

In [48]:
# Load movies metadata
print("Loading movies metadata...")
metadata_df = pd.read_csv(os.path.join(DATASET_PATH, 'movies_metadata.csv'),
                          dtype={'id': str},
                          low_memory=False)
metadata = metadata_df.copy()

print(f" Dataset loaded successfully")
print(f"Original row count: {len(metadata):,}")
print(f"Columns: {len(metadata.columns)}")

Loading movies metadata...
 Dataset loaded successfully
Original row count: 45,466
Columns: 24


In [49]:
# Check for missing data
print("Columns with missing data:\n")

missing_data = []
for column in metadata.columns:
    missing_count = metadata[column].isnull().sum()
    if missing_count > 0:
        missing_pct = (missing_count / len(metadata)) * 100
        missing_data.append({
            'Column': column,
            'Missing Count': missing_count,
            'Missing %': f"{missing_pct:.2f}%"
        })
        print(f"{column:30} | {missing_count:6} missing ({missing_pct:5.2f}%)")

print(f"\nTotal columns: {len(metadata.columns)}")
print(f"Columns with missing data: {len(missing_data)}")
print(f"Columns without missing data: {len(metadata.columns) - len(missing_data)}")

Columns with missing data:

belongs_to_collection          |  40972 missing (90.12%)
homepage                       |  37684 missing (82.88%)
imdb_id                        |     17 missing ( 0.04%)
original_language              |     11 missing ( 0.02%)
overview                       |    954 missing ( 2.10%)
popularity                     |      5 missing ( 0.01%)
poster_path                    |    386 missing ( 0.85%)
production_companies           |      3 missing ( 0.01%)
production_countries           |      3 missing ( 0.01%)
release_date                   |     87 missing ( 0.19%)
revenue                        |      6 missing ( 0.01%)
runtime                        |    263 missing ( 0.58%)
spoken_languages               |      6 missing ( 0.01%)
status                         |     87 missing ( 0.19%)
tagline                        |  25054 missing (55.10%)
title                          |      6 missing ( 0.01%)
video                          |      6 missing ( 0.01%)
vot

In [50]:
# Step 1: Fix malformed id field

# Convert id to numeric, coercing errors to NaN
metadata['id'] = pd.to_numeric(metadata['id'], errors='coerce')

# Count malformed ids
malformed_id_count = metadata['id'].isnull().sum()
print(f"Malformed ids found: {malformed_id_count}")

if malformed_id_count > 0:
    print("\nExamples of malformed rows:")
    print(metadata[metadata['id'].isnull()][['id', 'title', 'original_title']].head())

# Step 2: Drop rows with missing id or title
rows_before = len(metadata)
metadata = metadata.dropna(subset=['id', 'title'])
rows_dropped = rows_before - len(metadata)

print(f"Rows dropped: {rows_dropped}")
print(f"Remaining rows: {len(metadata):,}")

# Convert id to integer
metadata['id'] = metadata['id'].astype(int)
print(f"\n id field cleaned and converted to integer")

Malformed ids found: 3

Examples of malformed rows:
       id title                            original_title
19730 NaN   NaN  [{'iso_639_1': 'en', 'name': 'English'}]
29503 NaN   NaN      [{'iso_639_1': 'ja', 'name': '日本語'}]
35587 NaN   NaN  [{'iso_639_1': 'en', 'name': 'English'}]
Rows dropped: 6
Remaining rows: 45,460

 id field cleaned and converted to integer


In [51]:
# Step 3: Parse JSON columns

def parse_json_column(json_str):
    """Parse JSON string and extract 'name' fields"""
    try:
        if pd.isna(json_str):
            return []
        data = ast.literal_eval(json_str)
        if isinstance(data, list):
            return [item['name'] for item in data if 'name' in item]
        return []
    except:
        return []

# Parse genres
metadata['genres_list'] = metadata['genres'].apply(parse_json_column)

# Parse production_companies
metadata['production_companies_list'] = metadata['production_companies'].apply(parse_json_column)

# Parse production_countries
metadata['production_countries_list'] = metadata['production_countries'].apply(parse_json_column)

# Parse spoken_languages
metadata['spoken_languages_list'] = metadata['spoken_languages'].apply(parse_json_column)

print("\n JSON columns parsed successfully")
print("\nExample results:")
print(metadata[['title', 'genres_list', 'production_companies_list']].head(3))


 JSON columns parsed successfully

Example results:
              title                   genres_list                          production_companies_list
0         Toy Story   [Animation, Comedy, Family]                          [Pixar Animation Studios]
1           Jumanji  [Adventure, Fantasy, Family]  [TriStar Pictures, Teitler Film, Interscope Co...
2  Grumpier Old Men             [Romance, Comedy]                     [Warner Bros., Lancaster Gate]


In [52]:
# Step 4: Save cleaned dataset
columns_to_keep = [
    'id', 'title', 'original_title', 'release_date', 'runtime',
    'budget', 'revenue', 'popularity', 'vote_average', 'vote_count',
    'overview', 'tagline', 'imdb_id', 'original_language', 'status',
    'genres_list', 'production_companies_list', 
    'production_countries_list', 'spoken_languages_list'
]

cleaned_metadata = metadata[columns_to_keep].copy()

# Save to CSV
output_path = os.path.join(PROCESSED_DATA_DIR, 'cleaned_movies_metadata.csv')
cleaned_metadata.to_csv(output_path, index=False)

print(f"Cleaned metadata saved to: {output_path}")
print(f"Final dataset: {len(cleaned_metadata):,} rows, {len(cleaned_metadata.columns)} columns")

Cleaned metadata saved to: ./data/processed/cleaned_movies_metadata.csv
Final dataset: 45,460 rows, 19 columns


## 3. Clean Links Dataset

The links dataset maps MovieLens IDs to external IDs:
- `movieId`: MovieLens internal ID (used in ratings.csv)
- `imdbId`: IMDB identifier
- `tmdbId`: The Movie Database identifier (used in metadata, credits, keywords)

This is the **bridge** between ratings and movie metadata.

In [53]:
# Load links dataset
print("Loading links dataset...")
links_df = pd.read_csv(os.path.join(DATASET_PATH, 'links.csv'))
links = links_df.copy()

print(f" Dataset loaded successfully")
print(f"Columns: {list(links.columns)}")
print(f"\nSample data:")
print(links.head(10))

Loading links dataset...
 Dataset loaded successfully
Columns: ['movieId', 'imdbId', 'tmdbId']

Sample data:
   movieId  imdbId   tmdbId
0        1  114709    862.0
1        2  113497   8844.0
2        3  113228  15602.0
3        4  114885  31357.0
4        5  113041  11862.0
5        6  113277    949.0
6        7  114319  11860.0
7        8  112302  45325.0
8        9  114576   9091.0
9       10  113189    710.0


In [54]:
# Check for missing data
print("Columns with missing data:\n")
missing_data = []

for col in links.columns:
    missing_count = links[col].isnull().sum()
    if missing_count > 0:
        missing_pct = (missing_count / len(links)) * 100
        missing_data.append({
            'Column': col,
            'Missing Count': missing_count,
            'Missing %': f'{missing_pct:.2f}%'
        })
        print(f"{col:30} | {missing_count:6} missing ({missing_pct:5.2f}%)")

if len(missing_data) == 0:
    print(" No missing data found")
    
print(f"\nTotal columns: {len(links.columns)}")
print(f"Columns with missing data: {len(missing_data)}")

Columns with missing data:

tmdbId                         |    219 missing ( 0.48%)

Total columns: 3
Columns with missing data: 1


In [55]:
# Step 1: Drop rows with missing tmdbId (if any)
rows_before = len(links)
links = links.dropna(subset=['tmdbId'])
rows_dropped = rows_before - len(links)

print(f"Rows dropped due to missing tmdbId: {rows_dropped}")
print(f"Remaining rows: {len(links):,}")

# Step 2: Convert tmdbId to integer
links['tmdbId'] = links['tmdbId'].astype(int)

print(f"\n Data types converted")
print(f"\nCurrent data types:")
print(links.dtypes)

Rows dropped due to missing tmdbId: 219
Remaining rows: 45,624

 Data types converted

Current data types:
movieId    int64
imdbId     int64
tmdbId     int64
dtype: object


In [56]:
# Save cleaned links
output_path = os.path.join(PROCESSED_DATA_DIR, 'cleaned_links.csv')
links.to_csv(output_path, index=False)

print(f" Cleaned links saved to: {output_path}")
print(f"Final dataset: {len(links):,} rows, {len(links.columns)} columns")

 Cleaned links saved to: ./data/processed/cleaned_links.csv
Final dataset: 45,624 rows, 3 columns


## 4. Clean Credits Dataset

The credits dataset provides cast and crew information:
- `cast`: JSON array of actors (with character, order)
- `crew`: JSON array of crew members (with job, department)
- `id`: TMDB movie identifier


In [57]:
# Load credits dataset
print("Loading credits dataset...")
credits_df = pd.read_csv(os.path.join(DATASET_PATH, 'credits.csv'))
credits = credits_df.copy()

print(f" Dataset loaded: {len(credits):,} rows, {len(credits.columns)} columns")
print(f"Columns: {list(credits.columns)}")

Loading credits dataset...
 Dataset loaded: 45,476 rows, 3 columns
Columns: ['cast', 'crew', 'id']


In [58]:
# Parse cast and crew JSON columns
def parse_cast(cast_str, top_n=5):
    """Extract top N actor names from cast JSON"""
    try:
        if pd.isna(cast_str):
            return []
        cast_list = ast.literal_eval(cast_str)
        sorted_cast = sorted(cast_list, key=lambda x: x.get('order', 999))
        return [actor['name'] for actor in sorted_cast[:top_n] if 'name' in actor]
    except:
        return []

def parse_crew_directors(crew_str):
    """Extract director names from crew JSON"""
    try:
        if pd.isna(crew_str):
            return []
        crew_list = ast.literal_eval(crew_str)
        directors = [member['name'] for member in crew_list 
                    if member.get('job') == 'Director' and 'name' in member]
        return directors
    except:
        return []

credits['cast_list'] = credits['cast'].apply(parse_cast)

credits['director_list'] = credits['crew'].apply(parse_crew_directors)

print("\n JSON columns parsed successfully")
print("\nExample results:")
print(credits[['id', 'cast_list', 'director_list']].head(3))


 JSON columns parsed successfully

Example results:
      id                                          cast_list    director_list
0    862  [Tom Hanks, Tim Allen, Don Rickles, Jim Varney...  [John Lasseter]
1   8844  [Robin Williams, Jonathan Hyde, Kirsten Dunst,...   [Joe Johnston]
2  15602  [Walter Matthau, Jack Lemmon, Ann-Margret, Sop...  [Howard Deutch]


In [59]:
# Verify id exists in cleaned_movies_metadata
valid_movie_ids = set(cleaned_metadata['id'].unique())

print(f"Valid movie IDs in metadata: {len(valid_movie_ids):,}")

# Keep only credits for movies in metadata
rows_before = len(credits)
credits = credits[credits['id'].isin(valid_movie_ids)]
rows_dropped = rows_before - len(credits)

print(f"Rows dropped (id not in metadata): {rows_dropped}")
print(f"Remaining rows: {len(credits):,}")
print(f"Coverage: {(len(credits)/len(valid_movie_ids))*100:.2f}% of movies have credits")

Valid movie IDs in metadata: 45,430
Rows dropped (id not in metadata): 3
Remaining rows: 45,473
Coverage: 100.09% of movies have credits


In [60]:
# Save cleaned credits
columns_to_keep = ['id', 'cast_list', 'director_list']
cleaned_credits = credits[columns_to_keep].copy()

output_path = os.path.join(PROCESSED_DATA_DIR, 'cleaned_credits.csv')
cleaned_credits.to_csv(output_path, index=False)

print(f" Cleaned credits saved to: {output_path}")
print(f"Final dataset: {len(cleaned_credits):,} rows, {len(cleaned_credits.columns)} columns")

 Cleaned credits saved to: ./data/processed/cleaned_credits.csv
Final dataset: 45,473 rows, 3 columns


## 5. Clean Keywords Dataset

The keywords dataset provides movie tags/keywords:
- `id`: TMDB movie identifier
- `keywords`: JSON array of keyword objects


In [61]:
# Load keywords dataset
keywords_df = pd.read_csv(os.path.join(DATASET_PATH, 'keywords.csv'))
keywords = keywords_df.copy()

print(f" Dataset loaded: {len(keywords):,} rows, {len(keywords.columns)} columns")
print(f"Columns: {list(keywords.columns)}")

 Dataset loaded: 46,419 rows, 2 columns
Columns: ['id', 'keywords']


In [62]:
# Parse keywords JSON column
def parse_keywords(keywords_str):
    """Extract keyword names from JSON"""
    try:
        if pd.isna(keywords_str):
            return []
        keywords_list = ast.literal_eval(keywords_str)
        return [keyword['name'] for keyword in keywords_list if 'name' in keyword]
    except:
        return []

keywords['keywords_list'] = keywords['keywords'].apply(parse_keywords)

print("\n JSON column parsed successfully")
print("\nExample results:")
print(keywords[['id', 'keywords_list']].head(5))

# Show statistics
keyword_counts = keywords['keywords_list'].apply(len)
print(f"\nKeyword statistics:")
print(f"  - Average keywords per movie: {keyword_counts.mean():.2f}")
print(f"  - Median keywords per movie: {keyword_counts.median():.0f}")
print(f"  - Movies with no keywords: {(keyword_counts == 0).sum():,}")
print(f"  - Movies with keywords: {(keyword_counts > 0).sum():,}")


 JSON column parsed successfully

Example results:
      id                                      keywords_list
0    862  [jealousy, toy, boy, friendship, friends, riva...
1   8844  [board game, disappearance, based on children'...
2  15602  [fishing, best friend, duringcreditsstinger, o...
3  31357  [based on novel, interracial relationship, sin...
4  11862  [baby, midlife crisis, confidence, aging, daug...

Keyword statistics:
  - Average keywords per movie: 3.42
  - Median keywords per movie: 2
  - Movies with no keywords: 14,795
  - Movies with keywords: 31,624


In [63]:
# Verify id exists in cleaned_movies_metadata
rows_before = len(keywords)
keywords = keywords[keywords['id'].isin(valid_movie_ids)]
rows_dropped = rows_before - len(keywords)

print(f"Rows dropped (id not in metadata): {rows_dropped}")
print(f"Remaining rows: {len(keywords):,}")
print(f"Coverage: {(len(keywords)/len(valid_movie_ids))*100:.2f}% of movies have keywords")

Rows dropped (id not in metadata): 4
Remaining rows: 46,415
Coverage: 102.17% of movies have keywords


In [64]:
# Save cleaned keywords
columns_to_keep = ['id', 'keywords_list']
cleaned_keywords = keywords[columns_to_keep].copy()

output_path = os.path.join(PROCESSED_DATA_DIR, 'cleaned_keywords.csv')
cleaned_keywords.to_csv(output_path, index=False)

print(f" Cleaned keywords saved to: {output_path}")
print(f"Final dataset: {len(cleaned_keywords):,} rows, {len(cleaned_keywords.columns)} columns")

 Cleaned keywords saved to: ./data/processed/cleaned_keywords.csv
Final dataset: 46,415 rows, 2 columns


## Data Preprocessing Summary

All datasets have been cleaned and saved to `data/processed/`:

| Dataset | Status | Rows | Notes |
|---------|--------|------|-------|
|  cleaned_ratings.csv | Complete | ~26M | Core CF dataset |
|  cleaned_movies_metadata.csv | Complete | ~45K | Movie details with parsed JSON |
|  cleaned_links.csv | Complete | ~45K | ID mappings |
|  cleaned_credits.csv | Complete | ~45K | Top 5 actors + directors |
|  cleaned_keywords.csv | Complete | ~46K | Movie keywords/tags |

**Next Step**: Create train/test splits for model evaluation.

---

# Train/Test Split

We implement **two splitting methods** to compare their impact on model evaluation:

## Method 1: Random Split (80/20)
- **Approach**: Random sampling without temporal considerations
- **Use Case**: Baseline evaluation, ensures even distribution
- **Pros**: Simple, balanced user/movie coverage
- **Cons**: Does not reflect real-world temporal deployment

## Method 2: Temporal Split (80/20)
- **Approach**: Time-based split - train on past, test on future
- **Use Case**: Realistic evaluation, production-ready assessment
- **Pros**: Simulates real-world, prevents temporal leakage, industry standard
- **Cons**: May introduce cold-start challenges (new users/movies in test)

---

## Method 1: Random Split (80/20)

Split data randomly without considering temporal order.

In [65]:
# Load cleaned ratings
print("Loading cleaned ratings for random split...")
ratings_for_split = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'cleaned_ratings.csv'), low_memory=False)

print(f" Dataset loaded: {len(ratings_for_split):,} ratings")
print(f"Columns: {list(ratings_for_split.columns)}")

Loading cleaned ratings for random split...
 Dataset loaded: 26,024,289 ratings
Columns: ['userId', 'movieId', 'rating', 'timestamp']


In [66]:
# Perform random split
print("Performing random 80/20 train/test split...\n")

train_random, test_random = train_test_split(
    ratings_for_split,
    test_size=0.2,      # 20% for test
    random_state=42,    # For reproducibility
    shuffle=True        # Ensure random sampling
)

print(" Split completed successfully\n")
print(f"Training Set:")
print(f"  - Rows: {len(train_random):,} ({len(train_random)/len(ratings_for_split)*100:.2f}%)")
print(f"  - Users: {train_random['userId'].nunique():,}")
print(f"  - Movies: {train_random['movieId'].nunique():,}")

print(f"\nTest Set:")
print(f"  - Rows: {len(test_random):,} ({len(test_random)/len(ratings_for_split)*100:.2f}%)")
print(f"  - Users: {test_random['userId'].nunique():,}")
print(f"  - Movies: {test_random['movieId'].nunique():,}")

Performing random 80/20 train/test split...

 Split completed successfully

Training Set:
  - Rows: 20,819,431 (80.00%)
  - Users: 269,710
  - Movies: 43,326

Test Set:
  - Rows: 5,204,858 (20.00%)
  - Users: 253,107
  - Movies: 31,629


In [67]:
# Save random split datasets
output_dir = os.path.join(PROCESSED_DATA_DIR, 'random_split/')
os.makedirs(output_dir, exist_ok=True)

train_path = os.path.join(output_dir, 'train_ratings.csv')
test_path = os.path.join(output_dir, 'test_ratings.csv')

print("Saving random split datasets...\n")
train_random.to_csv(train_path, index=False)
test_random.to_csv(test_path, index=False)

train_size_mb = os.path.getsize(train_path) / (1024 * 1024)
test_size_mb = os.path.getsize(test_path) / (1024 * 1024)

print(f" Training set saved: {train_path} ({train_size_mb:.2f} MB)")
print(f" Test set saved: {test_path} ({test_size_mb:.2f} MB)")

Saving random split datasets...

 Training set saved: ./data/processed/random_split/train_ratings.csv (521.49 MB)
 Test set saved: ./data/processed/random_split/test_ratings.csv (130.37 MB)


## Method 2: Temporal Split (80/20)

Split data by time - train on older ratings, test on newer ratings.

**Why Temporal Split?**
- ✅ Simulates real-world deployment (predict future from past)
- ✅ Prevents temporal leakage (never trains on future)
- ✅ Industry standard (Netflix, Spotify, Amazon)
- ✅ Reveals cold-start challenges

In [68]:
# Analyze temporal distribution
print("Analyzing temporal distribution...\n")

ratings_for_split['datetime'] = pd.to_datetime(ratings_for_split['timestamp'], unit='s')
ratings_for_split['year'] = ratings_for_split['datetime'].dt.year

print(f"Temporal Range:")
print(f"  - Earliest rating: {ratings_for_split['datetime'].min()}")
print(f"  - Latest rating: {ratings_for_split['datetime'].max()}")
print(f"  - Time span: {(ratings_for_split['datetime'].max() - ratings_for_split['datetime'].min()).days} days")

# Calculate 80% split point
split_idx = int(len(ratings_for_split) * 0.8)
ratings_sorted = ratings_for_split.sort_values('timestamp')
split_timestamp = ratings_sorted.iloc[split_idx]['timestamp']
split_datetime = pd.to_datetime(split_timestamp, unit='s')

print(f"\n80/20 Split Point:")
print(f"  - Split date: {split_datetime}")
print(f"  - Training period: {ratings_sorted['datetime'].min()} to {split_datetime}")
print(f"  - Test period: {split_datetime} to {ratings_sorted['datetime'].max()}")

Analyzing temporal distribution...

Temporal Range:
  - Earliest rating: 1995-01-09 11:46:44
  - Latest rating: 2017-08-04 06:57:50
  - Time span: 8242 days

80/20 Split Point:
  - Split date: 2015-01-30 01:17:57
  - Training period: 1995-01-09 11:46:44 to 2015-01-30 01:17:57
  - Test period: 2015-01-30 01:17:57 to 2017-08-04 06:57:50


In [69]:
# Perform temporal split
print("Performing temporal 80/20 train/test split...\n")

# Sort by timestamp
ratings_sorted = ratings_for_split.sort_values('timestamp').reset_index(drop=True)
split_idx = int(len(ratings_sorted) * 0.8)

train_temporal = ratings_sorted.iloc[:split_idx].copy()
test_temporal = ratings_sorted.iloc[split_idx:].copy()

print(" Split completed successfully\n")

print(f"Training Set:")
print(f"  - Rows: {len(train_temporal):,} ({len(train_temporal)/len(ratings_for_split)*100:.2f}%)")
print(f"  - Period: {train_temporal['datetime'].min()} to {train_temporal['datetime'].max()}")
print(f"  - Users: {train_temporal['userId'].nunique():,}")
print(f"  - Movies: {train_temporal['movieId'].nunique():,}")

print(f"\nTest Set:")
print(f"  - Rows: {len(test_temporal):,} ({len(test_temporal)/len(ratings_for_split)*100:.2f}%)")
print(f"  - Period: {test_temporal['datetime'].min()} to {test_temporal['datetime'].max()}")
print(f"  - Users: {test_temporal['userId'].nunique():,}")
print(f"  - Movies: {test_temporal['movieId'].nunique():,}")

Performing temporal 80/20 train/test split...

 Split completed successfully

Training Set:
  - Rows: 20,819,431 (80.00%)
  - Period: 1995-01-09 11:46:44 to 2015-01-30 01:17:56
  - Users: 227,222
  - Movies: 25,636

Test Set:
  - Rows: 5,204,858 (20.00%)
  - Period: 2015-01-30 01:17:57 to 2017-08-04 06:57:50
  - Users: 48,464
  - Movies: 42,721


In [70]:
# Cold-start analysis
print("Analyzing cold-start scenarios...\n")

train_users = set(train_temporal['userId'].unique())
test_users = set(test_temporal['userId'].unique())
new_users_in_test = test_users - train_users

train_movies = set(train_temporal['movieId'].unique())
test_movies = set(test_temporal['movieId'].unique())
new_movies_in_test = test_movies - train_movies

print(f"User Analysis:")
print(f"  - Train users: {len(train_users):,}")
print(f"  - Test users: {len(test_users):,}")
print(f"  - NEW users in test: {len(new_users_in_test):,} ({len(new_users_in_test)/len(test_users)*100:.2f}%)")

print(f"\nMovie Analysis:")
print(f"  - Train movies: {len(train_movies):,}")
print(f"  - Test movies: {len(test_movies):,}")
print(f"  - NEW movies in test: {len(new_movies_in_test):,} ({len(new_movies_in_test)/len(test_movies)*100:.2f}%)")

print(f"\nNote: Cold-start items reflect realistic production scenarios.")

Analyzing cold-start scenarios...

User Analysis:
  - Train users: 227,222
  - Test users: 48,464
  - NEW users in test: 43,674 (90.12%)

Movie Analysis:
  - Train movies: 25,636
  - Test movies: 42,721
  - NEW movies in test: 19,479 (45.60%)

Note: Cold-start items reflect realistic production scenarios.


In [71]:
# Save temporal split datasets (remove datetime columns)
columns_to_save = ['userId', 'movieId', 'rating', 'timestamp']

train_temporal_final = train_temporal[columns_to_save].copy()
test_temporal_final = test_temporal[columns_to_save].copy()

output_dir = os.path.join(PROCESSED_DATA_DIR, 'temporal_split/')
os.makedirs(output_dir, exist_ok=True)

train_path = os.path.join(output_dir, 'train_ratings.csv')
test_path = os.path.join(output_dir, 'test_ratings.csv')

print("Saving temporal split datasets...\n")
train_temporal_final.to_csv(train_path, index=False)
test_temporal_final.to_csv(test_path, index=False)

train_size_mb = os.path.getsize(train_path) / (1024 * 1024)
test_size_mb = os.path.getsize(test_path) / (1024 * 1024)

print(f" Training set saved: {train_path} ({train_size_mb:.2f} MB)")
print(f" Test set saved: {test_path} ({test_size_mb:.2f} MB)")

Saving temporal split datasets...

 Training set saved: ./data/processed/temporal_split/train_ratings.csv (517.09 MB)
 Test set saved: ./data/processed/temporal_split/test_ratings.csv (134.77 MB)


## Train/Test Split Summary

### Method 1: Random Split (80/20)
**Output Location**: `data/processed/random_split/`
-  `train_ratings.csv` - Training set (~80% random sample)
-  `test_ratings.csv` - Test set (~20% random sample)

**Characteristics**:
- Random sampling without temporal order
- Balanced user/movie distribution

### Method 2: Temporal Split (80/20)
**Output Location**: `data/processed/temporal_split/`
-  `train_ratings.csv` - Training set (older 80% by timestamp)
-  `test_ratings.csv` - Test set (newer 20% by timestamp)

**Characteristics**:
- Time-based split (train on past, test on future)
- Prevents temporal leakage
- Includes cold-start scenarios

